# 01 · Metadatos v2

Lee solo cabeceras DICOM (todos los archivos de cada serie, sin píxeles) de train y test y deriva por serie: plano por
orientación, ponderación, supresión grasa, espaciado, centro físico y lateralidad. Por estudio: lado, centro y los 6
slots de `docs/pasos/paso3_entrada_v2.md`. Valida el lado contra el informe. CPU, sin GPU, ~15–30 min.

In [ ]:
# CONFIG
EXP = "exp009_preparacion_v2"
PADRE = "exp001"
RAW_ROOT = "/workspace/data/raw"
OUT_DIR = "/workspace/data/cache/meta_v2"
MAX_STUDIES = None
N_THREADS = 16
SIDE_MM = 15.0
MIN_PRECISION = 0.95

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from multiprocessing.pool import ThreadPool
from pathlib import Path
import re
import sys
import time

import numpy as np
import pandas as pd
import pydicom

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "knee").exists())
sys.path.insert(0, str(repo))
from src.knee.config import current_user
from src.knee.experiments import git_info, guardar_metricas

raw = Path(RAW_ROOT)
out = Path(OUT_DIR) / ("smoke" if MAX_STUDIES else "")
out.mkdir(parents=True, exist_ok=True)
results = repo / "results" / EXP
assert (repo / "experiments" / EXP / "SPEC.md").exists(), "Falta la SPEC"
print(f"Salida: {out}")

## Series de train y test

In [ ]:
lists = []
for split in ["train", "test"]:
    df = pd.read_csv(raw / f"{split}_series.csv")
    df["split"] = split
    lists.append(df)
series = pd.concat(lists, ignore_index=True).rename(columns={"StudyInstanceUID": "study", "SeriesInstanceUID": "series"})
if MAX_STUDIES:
    keep = series.drop_duplicates("study").groupby("split").head(MAX_STUDIES).study
    series = series[series.study.isin(keep)].reset_index(drop=True)
print(series.groupby("split").agg(estudios=("study", "nunique"), series=("series", "count")))

## Cabeceras de todos los archivos

In [ ]:
TAGS = ["ImageOrientationPatient", "ImagePositionPatient", "PixelSpacing", "SliceThickness", "RepetitionTime",
        "EchoTime", "InversionTime", "ScanOptions", "ScanningSequence", "SequenceVariant", "SequenceName",
        "Manufacturer", "MagneticFieldStrength", "SeriesDescription", "SeriesNumber", "ImageType", "Laterality",
        "ImageLaterality", "BodyPartExamined", "PatientPosition", "InstanceNumber", "Rows", "Columns", "ProtocolName"]


def as_list(value):
    if value is None:
        return []
    if isinstance(value, (str, bytes)) or not hasattr(value, "__iter__"):
        return [str(value)]
    return [str(v) for v in value]


def as_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def first(values):
    values = [v for v in values if v not in (None, "")]
    return Counter(values).most_common(1)[0][0] if values else None


def read_series_meta(row):
    folder = raw / f"{row.split}_series" / row.study / row.series
    files = sorted(folder.glob("*.dcm"))
    heads, bad = [], 0
    for path in files:
        try:
            heads.append(pydicom.dcmread(path, stop_before_pixels=True, specific_tags=TAGS))
        except Exception:
            bad += 1
    meta = {"study": row.study, "series": row.series, "n_files": len(files), "n_read_errors": bad}
    if not heads:
        return meta
    get = lambda key: [getattr(h, key, None) for h in heads]
    iops = [np.asarray(v, dtype=float) if v is not None and len(v) == 6 else None for v in get("ImageOrientationPatient")]
    ipps = [np.asarray(v, dtype=float) if v is not None and len(v) == 3 else None for v in get("ImagePositionPatient")]
    pss = [np.asarray(v, dtype=float) if v is not None and len(v) == 2 else None for v in get("PixelSpacing")]
    rows_, cols_ = get("Rows"), get("Columns")
    inst = [as_float(v) for v in get("InstanceNumber")]
    ok = [i for i in range(len(heads)) if iops[i] is not None and ipps[i] is not None and pss[i] is not None]
    meta.update({
        "manufacturer": first(get("Manufacturer")),
        "field_t": as_float(first(get("MagneticFieldStrength"))),
        "description": first(get("SeriesDescription")),
        "protocol": first(get("ProtocolName")),
        "series_number": as_float(first(get("SeriesNumber"))),
        "image_type": "|".join(as_list(heads[0].get("ImageType"))),
        "scan_options": "|".join(sorted({t for h in heads for t in as_list(h.get("ScanOptions")) if t})),
        "scanning_sequence": "|".join(sorted({t for h in heads for t in as_list(h.get("ScanningSequence")) if t})),
        "sequence_variant": "|".join(sorted({t for h in heads for t in as_list(h.get("SequenceVariant")) if t})),
        "sequence_name": first([str(v) for v in get("SequenceName") if v]),
        "tr": np.nanmedian([as_float(v) for v in get("RepetitionTime")] + [np.nan]),
        "te": np.nanmedian([as_float(v) for v in get("EchoTime")] + [np.nan]),
        "ti": np.nanmedian([as_float(v) for v in get("InversionTime")] + [np.nan]),
        "slice_thickness": np.nanmedian([as_float(v) for v in get("SliceThickness")] + [np.nan]),
        "laterality_tag": "|".join(sorted({str(v).strip().upper() for k in ["Laterality", "ImageLaterality"]
                                           for v in get(k) if v and str(v).strip()})),
        "body_part": first(get("BodyPartExamined")),
        "patient_position": first(get("PatientPosition")),
        "rows": first(rows_), "cols": first(cols_),
        "n_geom": len(ok),
    })
    if not ok:
        return meta
    ref = Counter(tuple(np.round(iops[i], 2)) for i in ok).most_common(1)[0][0]
    ref = np.asarray(ref)
    dev = np.array([np.abs(iops[i] - ref).max() for i in ok])
    same = [i for i, d in zip(ok, dev) if d < 0.05]
    row_dir, col_dir = np.mean([iops[i][:3] for i in same], 0), np.mean([iops[i][3:] for i in same], 0)
    normal = np.cross(row_dir, col_dir)
    normal /= np.linalg.norm(normal)
    pos = np.array([ipps[i] @ normal for i in same])
    order = np.sort(pos)
    diffs = np.diff(order)
    centers = [ipps[i] + iops[i][:3] * pss[i][1] * (cols_[i] - 1) / 2 + iops[i][3:] * pss[i][0] * (rows_[i] - 1) / 2
               for i in same]
    ranks = lambda a: np.argsort(np.argsort(a))
    inst_same = np.array([inst[i] for i in same])
    corr = (np.corrcoef(ranks(pos), ranks(inst_same))[0, 1]
            if len(same) >= 3 and np.isfinite(inst_same).all() and np.ptp(pos) > 0 else np.nan)
    ps = np.median([pss[i] for i in same], 0)
    meta.update({
        "nx": normal[0], "ny": normal[1], "nz": normal[2],
        "iop_max_dev": float(dev.max()), "n_orient_outliers": len(ok) - len(same),
        "n_slices": len(same), "n_unique_pos": len(np.unique(np.round(pos, 1))),
        "slice_spacing": float(np.median(diffs)) if len(diffs) else np.nan,
        "slice_spacing_max": float(diffs.max()) if len(diffs) else np.nan,
        "extent_mm": float(np.ptp(pos)),
        "ps_row": ps[0], "ps_col": ps[1],
        "fov_mm": float(max(ps[0] * (first(rows_) or 0), ps[1] * (first(cols_) or 0))),
        "cx": float(np.mean([c[0] for c in centers])), "cy": float(np.mean([c[1] for c in centers])),
        "cz": float(np.mean([c[2] for c in centers])),
        "inst_pos_corr": corr,
    })
    return meta


def safe_meta(row):
    try:
        return read_series_meta(row)
    except Exception as error:
        return {"study": row.study, "series": row.series, "error": repr(error)[:200]}

In [ ]:
start = time.time()
rows, done = [], 0
with ThreadPool(N_THREADS) as pool:
    for meta in pool.imap_unordered(safe_meta, series.itertuples(index=False), chunksize=4):
        rows.append(meta)
        done += 1
        if done % 2000 == 0:
            print(f"{done:,}/{len(series):,} series · {time.time() - start:.0f} s", flush=True)
read_seconds = time.time() - start
meta = series.merge(pd.DataFrame(rows), on=["study", "series"], how="left")
print(f"{len(meta):,} series · {meta.n_files.sum():,} archivos · {read_seconds / 60:.1f} min")
print(f"Errores de lectura: {int(meta.n_read_errors.sum())} archivos; series sin geometría: {int((meta.n_geom.fillna(0) == 0).sum())}")

## Plano, ponderación, supresión grasa y descartes

In [ ]:
normal = meta[["nx", "ny", "nz"]].abs()
meta["plane_max"] = normal.max(axis=1)
meta["plane"] = normal.fillna(-1).idxmax(axis=1).map({"nx": "Sagittal", "ny": "Coronal", "nz": "Axial"})
meta.loc[meta.plane_max < 0.8, "plane"] = "Oblique"
meta.loc[meta.plane_max.isna(), "plane"] = "Unknown"


def weighting(r):
    seq = r.scanning_sequence if isinstance(r.scanning_sequence, str) else ""
    if (100 <= r.ti <= 250) or ("IR" in seq.split("|") and not r.ti >= 300):
        return "STIR"
    if not (np.isfinite(r.tr) and np.isfinite(r.te)):
        return "unknown"
    if r.tr < 1000 and r.te < 30:
        return "T1"
    if r.tr >= 1500:
        return "PD" if r.te < 60 else "T2"
    return "other"


WEIGHT_TEXT = [("STIR", r"stir|tirm"), ("PD", r"pd|(?<![a-z])dp(?![a-z])|rho|proton"),
               ("T2", r"t2(?![0-9*])"), ("T1", r"t1(?![0-9])")]


def weighting_text(text):
    text = str(text or "").lower()
    return next((name for name, pattern in WEIGHT_TEXT if re.search(pattern, text)), None)


FS_TOKENS = {"FS", "SFS", "SPAIR", "SPIR", "WE", "CHESS", "FATSAT", "FAT_SAT"}
FS_TEXT = re.compile(r"fs(?!e|pgr)|fat.?sat|spair|spir|stir|tirm|(?<![a-z])we(?![a-z])|water|dixon.?w|f\.s\.", re.I)


def fat_sat(r):
    if FS_TOKENS & set(str(r.scan_options).split("|")):
        return 1, "scan_options"
    if r.weighting == "STIR":
        return 1, "stir"
    text = f"{r.description or ''} {r.protocol or ''}"
    if FS_TEXT.search(text):
        return 1, "text"
    return int(r.Fat_Suppression), "csv"


meta["weighting"] = meta.apply(weighting, axis=1)
meta["weighting_src"] = np.where(meta.weighting.isin(["unknown", "other"]), None, "trte")
by_text = meta.description.fillna("").map(weighting_text)
use_text = meta.weighting_src.isna() & by_text.notna()
meta.loc[use_text, "weighting"], meta.loc[use_text, "weighting_src"] = by_text[use_text], "text"
meta["weighting_src"] = meta.weighting_src.fillna("none")
meta[["fs", "fs_src"]] = meta.apply(fat_sat, axis=1, result_type="expand")
meta["fluid"] = meta.weighting.map({"PD": 1, "T2": 1, "STIR": 1, "T1": 0})
meta["fluid_src"] = np.where(meta.fluid.notna(), meta.weighting_src, "csv")
meta["fluid"] = meta.fluid.fillna(meta.Fluid_Sensitive).astype(int)
meta["is_original"] = meta.image_type.fillna("").str.startswith("ORIGINAL")
LOC_TEXT = re.compile(r"loc|scout|survey|3.?pl|tri.?pilot|localizer|localiser|plan(?!ar)", re.I)
meta["localizer"] = (meta.image_type.fillna("").str.contains("LOCALIZER")
                     | meta.description.fillna("").str.contains(LOC_TEXT)
                     | (meta.n_orient_outliers.fillna(0) > 0))
meta["order_reversed"] = meta.inst_pos_corr < 0
meta["eligible"] = ~meta.localizer & (meta.n_slices.fillna(0) >= 5) & meta.plane.isin(["Sagittal", "Coronal", "Axial"])

known = meta.plane.isin(["Sagittal", "Coronal", "Axial"])
print(f"Plano por IOP: {known.mean():.2%} · acuerdo con csv: {(meta.plane == meta.Anatomical_Plane)[known].mean():.2%}")
display(pd.crosstab(meta.Anatomical_Plane, meta.plane, margins=True))
display(pd.crosstab([meta.weighting_src, meta.weighting], meta.Fluid_Sensitive, margins=True))
print(f"csv Fluid_Sensitive == csv Fat_Suppression: {(meta.Fluid_Sensitive == meta.Fat_Suppression).mean():.2%}")
display(pd.crosstab(meta.fs_src, meta.fs == meta.Fat_Suppression, margins=True))
print(f"FS v2 = csv: {(meta.fs == meta.Fat_Suppression).mean():.2%} · líquido v2 = csv: {(meta.fluid == meta.Fluid_Sensitive).mean():.2%}")
print(f"Localizadores: {int(meta.localizer.sum())} · < 5 cortes: {int((meta.n_slices.fillna(0) < 5).sum())} · "
      f"elegibles: {meta.eligible.mean():.2%} · orden invertido vs InstanceNumber: {meta.order_reversed.mean():.2%}")
display(meta[["ps_row", "slice_spacing", "slice_thickness", "n_slices", "fov_mm"]].describe().round(2))

## Lateralidad: tag → texto → geometría → voto del estudio

In [ ]:
TEXT_L = {"l", "lt", "left", "izq", "izqda", "izda", "izdo", "izquierda", "izquierdo", "links", "sx", "gauche",
          "esq", "esquerdo", "esquerda", "lijevo", "lijevi", "levo", "lewe", "lewy"}
TEXT_R = {"r", "rt", "right", "der", "dcha", "dcho", "derecha", "derecho", "rechts", "dx", "droit", "droite",
          "direito", "direita", "desno", "desni", "prawe", "prawy"}


def side_tag(value):
    sides = {"L" if v in ("L", "LEFT") else "R" if v in ("R", "RIGHT") else None for v in str(value).split("|")}
    sides.discard(None)
    return sides.pop() if len(sides) == 1 else None


def side_text(r):
    tokens = set(re.split(r"[^a-z0-9]+", f"{r.description or ''} {r.protocol or ''}".lower()))
    left, right = bool(tokens & TEXT_L), bool(tokens & TEXT_R)
    return "L" if left and not right else "R" if right and not left else None


meta["side_tag"] = meta.laterality_tag.fillna("").map(side_tag)
meta["side_text"] = meta.apply(side_text, axis=1)
meta["side_geom"] = np.where(meta.cx > SIDE_MM, "L", np.where(meta.cx < -SIDE_MM, "R", None))
meta.loc[meta.cx.isna(), "side_geom"] = None


def majority(values):
    counts = Counter(v for v in values if v in ("L", "R"))
    if not counts or counts["L"] == counts["R"]:
        return None
    return counts.most_common(1)[0][0]


studies = meta.groupby("study").agg(split=("split", "first"), n_series=("series", "count"))
for method in ["tag", "text", "geom"]:
    studies[f"study_{method}"] = meta.groupby("study")[f"side_{method}"].agg(majority)
    studies[f"conflict_{method}"] = meta.groupby("study")[f"side_{method}"].agg(lambda s: s.isin(["L", "R"]).any() and s.nunique() > 1)
display(studies[[c for c in studies if c.startswith("conflict")]].mean().rename("estudios con series en conflicto"))

## Lado del informe (solo train) y precisión por método

In [ ]:
REPORT_L = re.compile(r"\b(left|izquierd[ao]s?|izq|izda|izdo|esquerd[ao]s?|links|linke[nrs]?|gauche|sinistr[oa]|"
                      r"lew[eay]|lewego|lewym|lewej|lijev\w*|levé|levého|levém|levý|levá|levo|levog|ľav\w*|sol|"
                      r"αριστερ\w*|лев(?:ый|ого|ое|ом|ая|ой|ую)|ляв\w*|vänster|venstre|stâng\w*)\b|ginocchio\s+sx|左|좌측|왼쪽", re.I)
REPORT_R = re.compile(r"\b(right|derech[ao]s?|dcha|dcho|direit[ao]s?|rechts|rechte[nrs]?|droite?|destr[oa]|"
                      r"praw[eay]|prawego|prawym|prawej|desn\w*|pravé|pravého|pravém|pravý|pravá|pravej|pravom|sağ|"
                      r"δεξι\w*|прав(?:ый|ого|ое|ом|ая|ой|ую)|дясн\w*|höger|høyre|højre|drept\w*)\b|ginocchio\s+dx|右|우측|오른쪽", re.I)

train = pd.read_csv(raw / "train.csv", encoding="utf-8").set_index("StudyInstanceUID")
reports = train.Report.reindex(studies.index).fillna("")


def report_side(text):
    n_left, n_right = len(REPORT_L.findall(text)), len(REPORT_R.findall(text))
    if n_left == n_right == 0:
        return "none"
    if n_left >= 2 * max(n_right, 1) or (n_left and not n_right):
        return "L"
    if n_right >= 2 * max(n_left, 1) or (n_right and not n_left):
        return "R"
    return "both"


studies["report_side"] = reports.map(report_side).where(studies.split == "train")
print(studies[studies.split == "train"].report_side.value_counts(dropna=False))

valid = studies[studies.report_side.isin(["L", "R"])]
agreement = {}
for method in ["tag", "text", "geom"]:
    known = valid[f"study_{method}"].notna()
    agreement[method] = {
        "coverage_all": float(studies[f"study_{method}"].notna().mean()),
        "n_checked": int(known.sum()),
        "precision": float((valid.loc[known, f"study_{method}"] == valid.loc[known, "report_side"]).mean()) if known.any() else None,
    }
display(pd.DataFrame(agreement).T)

In [ ]:
strong = [m for m in ["tag", "text", "geom"] if (agreement[m]["precision"] or 0) >= MIN_PRECISION]
weak = [m for m in ["tag", "text", "geom"] if m not in strong]
print(f"Métodos fuertes: {strong} · débiles (solo si el estudio no tiene nada más): {weak}")
studies["study_side"], studies["study_side_src"] = "unknown", "unknown"
for method in (strong + weak)[::-1]:
    has = studies[f"study_{method}"].notna()
    studies.loc[has, "study_side"] = studies.loc[has, f"study_{method}"]
    studies.loc[has, "study_side_src"] = method

meta["side"], meta["side_src"] = None, None
for method in strong[::-1]:
    has = meta[f"side_{method}"].notna()
    meta.loc[has, "side"], meta.loc[has, "side_src"] = meta.loc[has, f"side_{method}"], method
fill = meta.side.isna()
meta.loc[fill, "side"] = meta.loc[fill, "study"].map(studies.study_side)
meta.loc[fill, "side_src"] = "study"
meta.loc[meta.side == "unknown", "side_src"] = "unknown"

known = studies.study_side != "unknown"
check = studies.report_side.isin(["L", "R"]) & known
print(f"Estudios con lado: {known.mean():.2%} · precisión vs informe: "
      f"{(studies.study_side == studies.report_side)[check].mean():.2%} (n = {int(check.sum())})")
display(pd.crosstab(studies.study_side_src[check], (studies.study_side == studies.report_side)[check]).rename(columns={True: "acuerdo", False: "desacuerdo"}))
both = studies.study_tag.notna() & studies.study_geom.notna()
tag_geom = float((studies.study_tag == studies.study_geom)[both].mean())
print(f"Tag y geometría coinciden en {tag_geom:.2%} de {int(both.sum())} estudios con ambos")
display(meta.side_src.value_counts(normalize=True).round(4))

## Slots

In [ ]:
SLOTS = {
    "slot1_sag_fluid_fs": [("Sagittal", lambda m: (m.fluid == 1) & (m.fs == 1))],
    "slot2_cor_fluid_fs": [("Coronal", lambda m: (m.fluid == 1) & (m.fs == 1))],
    "slot3_ax_fluid_fs": [("Axial", lambda m: (m.fluid == 1) & (m.fs == 1)),
                          ("Axial", lambda m: m.weighting.isin(["PD", "T2"]))],
    "slot4_sag_pdt2": [("Sagittal", lambda m: m.weighting.isin(["PD", "T2"]) & (m.fs == 0))],
    "slot5_cor_t1": [("Coronal", lambda m: (m.weighting == "T1") & (m.fs == 0)),
                     ("Coronal", lambda m: (m.weighting == "PD") & (m.fs == 0))],
    "slot6_sag_t1": [("Sagittal", lambda m: (m.weighting == "T1") & (m.fs == 0))],
}

pool = meta[meta.eligible].copy()
pool["ps_min"] = pool[["ps_row", "ps_col"]].min(axis=1).round(3)
pool["in_range"] = pool.n_slices.between(15, 60)
pool = pool.sort_values(["is_original", "ps_min", "in_range", "series_number"], ascending=[False, True, False, False])
for slot, rules in SLOTS.items():
    studies[slot], studies[f"{slot}_fallback"] = None, False
    for level, (plane, rule) in enumerate(rules):
        hits = pool[(pool.plane == plane) & rule(pool)].drop_duplicates("study").set_index("study").series
        empty = studies[slot].isna() & studies.index.isin(hits.index)
        studies.loc[empty, slot] = hits.reindex(studies.index[empty]).values
        studies.loc[empty, f"{slot}_fallback"] = level > 0
slot_cols = list(SLOTS)
studies["n_slots"] = studies[slot_cols].notna().sum(axis=1)
coverage = studies.groupby("split")[slot_cols].agg(lambda s: s.notna().mean()).T
coverage["fallback_train"] = [studies.loc[studies.split == "train", f"{s}_fallback"].mean() for s in slot_cols]
display(coverage.round(4))
display(studies.groupby("split").n_slots.value_counts().unstack(0).fillna(0).astype(int))

## Centro del estudio e idioma aproximado del informe

In [ ]:
centers = meta[meta.eligible].groupby("study")[["cx", "cy", "cz"]].median()
studies = studies.join(centers.add_prefix("study_"))

LANGS = {
    "el": r"[α-ω]", "bg_ru": r"[а-я]", "tr": r"[ığş]|\bdiz\b", "de": r"\b(der|die|und|nicht|kein\w*|im)\b",
    "nl": r"\b(het|van|een|geen|bevindingen)\b", "es": r"\b(el|la|los|rodilla|sin|menisco)\b",
    "pt": r"\b(joelho|não|com|sem|menisco)\b", "it": r"\b(ginocchio|della|il|non)\b",
    "fr": r"\b(le|les|des|du|genou)\b", "hr": r"\b(koljen\w*|je|te|uz)\b", "pl": r"\b(kolan\w*|jest|bez)\b",
    "en": r"\b(the|of|and|knee|normal)\b",
}


def language(text):
    text = text.lower()
    scores = {k: len(re.findall(p, text)) for k, p in LANGS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] else "none"


studies["lang"] = reports.map(language).where(studies.split == "train", "test")
by_lang = studies[studies.split == "train"].groupby("lang").agg(
    estudios=("n_series", "size"), slot1=("slot1_sag_fluid_fs", lambda s: s.notna().mean()),
    slot2=("slot2_cor_fluid_fs", lambda s: s.notna().mean()), slot3=("slot3_ax_fluid_fs", lambda s: s.notna().mean()),
    lado=("study_side", lambda s: (s != "unknown").mean()),
    lado_ok=("study_side", lambda s: (s == studies.loc[s.index, "report_side"]).sum()
             / max(1, (studies.loc[s.index, "report_side"].isin(["L", "R"]) & (s != "unknown")).sum())))
display(by_lang.round(3).sort_values("estudios", ascending=False))

## Guardar

In [ ]:
meta.to_csv(out / "series_meta_v2.csv", index=False)
studies.reset_index().to_csv(out / "study_meta_v2.csv", index=False)

train_st = studies[studies.split == "train"]
check = train_st.report_side.isin(["L", "R"]) & (train_st.study_side != "unknown")
known = meta.plane.isin(["Sagittal", "Coronal", "Axial"])
metrics = {
    "schema_version": 1, "exp": EXP, "author": current_user(), "parent": PADRE, "git": git_info(repo),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(), "phase": "prep" if not MAX_STUDIES else "smoke",
    "params": {"MAX_STUDIES": MAX_STUDIES, "N_THREADS": N_THREADS, "SIDE_MM": SIDE_MM, "MIN_PRECISION": MIN_PRECISION},
    "outputs": [str(out / "series_meta_v2.csv"), str(out / "study_meta_v2.csv")],
    "read_minutes": read_seconds / 60,
    "n_studies": studies.groupby("split").size().to_dict(),
    "n_series": meta.groupby("split").size().to_dict(),
    "n_files": int(meta.n_files.sum()), "n_read_errors": int(meta.n_read_errors.sum()),
    "series_without_geometry": int((meta.n_geom.fillna(0) == 0).sum()),
    "plane_iop_coverage": float(known.mean()),
    "plane_counts": meta.plane.value_counts().to_dict(),
    "plane_agree_csv": float((meta.plane == meta.Anatomical_Plane)[known].mean()),
    "weighting_counts": meta.weighting.value_counts().to_dict(),
    "weighting_src_counts": meta.weighting_src.value_counts().to_dict(),
    "csv_fluid_equals_csv_fs": float((meta.Fluid_Sensitive == meta.Fat_Suppression).mean()),
    "side_tag_vs_geom_agree": tag_geom, "side_tag_vs_geom_n": int(both.sum()),
    "fs_src_counts": meta.fs_src.value_counts().to_dict(),
    "fs_agree_csv": float((meta.fs == meta.Fat_Suppression).mean()),
    "fluid_agree_csv": float((meta.fluid == meta.Fluid_Sensitive).mean()),
    "localizers": int(meta.localizer.sum()), "eligible_frac": float(meta.eligible.mean()),
    "order_reversed_frac": float(meta.order_reversed.mean()),
    "median_ps_mm": float(meta.ps_row.median()), "median_slice_spacing_mm": float(meta.slice_spacing.median()),
    "report_side_counts": train_st.report_side.value_counts().to_dict(),
    "side_methods": agreement, "side_strong": strong, "side_weak": weak,
    "study_side_coverage": float((studies.study_side != "unknown").mean()),
    "study_side_src_counts": studies.study_side_src.value_counts().to_dict(),
    "study_side_precision": float((train_st.study_side == train_st.report_side)[check].mean()),
    "study_side_n_checked": int(check.sum()),
    "slot_coverage": {split: {s: float(v) for s, v in coverage[split].items()} for split in coverage.columns if split in ("train", "test")},
    "slot_fallback_train": {s: float(v) for s, v in coverage.fallback_train.items()},
}
print(guardar_metricas(results if not MAX_STUDIES else results / "smoke", metrics))

## Verdict

Copia los números de `results/exp009_preparacion_v2/metrics.json` a la sección 3 de la SPEC y a `docs/EXPERIMENTOS.md`.